In [80]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [81]:
BASE_DIR = Path.cwd().parent

STAGE1_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage1"
    / "stage1_model_keys.csv"
)

print("Current directory:", Path.cwd())
print("Stage 1 file:", STAGE1_PATH.resolve())
print("File exists:", STAGE1_PATH.exists())

Current directory: c:\Users\MG\Documents\GitHub\DEPI-GRADUATION-PROJECT\DEPI-GRADUATION-PROJECT\notebooks
Stage 1 file: C:\Users\MG\Documents\GitHub\DEPI-GRADUATION-PROJECT\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_model_keys.csv
File exists: True


In [82]:
if not STAGE1_PATH.exists():
    raise FileNotFoundError(
        f"Stage 1 file not found: {STAGE1_PATH.resolve()}"
    )

stage1_df = pd.read_csv(STAGE1_PATH)

print("Shape:", stage1_df.shape)
print("Columns:", stage1_df.columns.tolist())

display(stage1_df.head())

Shape: (5258, 36)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'model_key', 'match_rule', 'confidence']


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,loc_city,price_missing,price_suspicious,price_very_high,title_normalized,title_punct_normalized,title_group,model_key,match_rule,confidence
0,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,مدينتي,False,False,False,honor magic v5,honor magic v5,honor magic v5,honor magic v5,honor:rule_04,NaN
1,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,القاهرة الجديدة,False,False,False,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,xiaomi 17 ultra,xiaomi:rule_17,NaN
2,Apple - iPhone 17 Pro,75000.0,محور 26 يوليو، 6 اكتوبر,/ad/apple-iphone-17-pro-ID208973056.html,208973056,legacy,apple,NaN,NaN,NaN,...,6 اكتوبر,False,False,False,apple iphone17pro,apple iphone17pro,apple iphone17pro,apple iphone 17 pro,apple:model_pattern_06,1.0
3,Honor 500 pro,35000.0,6 اكتوبر، الجيزة,/ad/honor-500-pro-ID208029599.html,208029599,legacy,honor,NaN,NaN,NaN,...,الجيزة,False,False,False,honor500pro,honor500pro,honor500pro,honor 500 pro,honor:rule_07,NaN
4,آبل - آيفون 15 برو ماكس,58000.0,ستيلا هليوبوليس، مدينة الشروق,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,208989761,legacy,apple,NaN,NaN,NaN,...,مدينة الشروق,False,False,False,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,apple iphone 15 pro max,apple:model_pattern_02,1.0


In [83]:
display(stage1_df.dtypes.to_frame("dtype"))
display(stage1_df.isna().sum().to_frame("missing_values"))

,dtype
title,str
price,float64
location,str
ad_url,str
ad_id,int64
run_id,str
brand,str
storage_gb,float64
ram_gb,float64
storage_src,str


,missing_values
title,0
price,0
location,4
ad_url,0
ad_id,0
run_id,0
brand,279
storage_gb,4891
ram_gb,5146
storage_src,4891


In [84]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

for col in known_flags:
    print(f"{col}: {'EXISTS' if col in stage1_df.columns else 'MISSING'}")

storage_known: MISSING
ram_known: MISSING
condition_known: MISSING


In [85]:
spec_cols = ["storage_gb", "ram_gb", "condition"]

display(
    pd.DataFrame({
        "dtype": stage1_df[spec_cols].dtypes,
        "missing_count": stage1_df[spec_cols].isna().sum(),
        "known_count": stage1_df[spec_cols].notna().sum(),
        "known_percentage": (
            stage1_df[spec_cols].notna().mean() * 100
        ).round(2),
    })
)

,dtype,missing_count,known_count,known_percentage
storage_gb,float64,4891,367,6.98
ram_gb,float64,5146,112,2.13
condition,str,4975,283,5.38


creating known flags  (storage,ram,condition)

In [86]:
stage1_df["storage_known"] = stage1_df["storage_gb"].notna()
stage1_df["ram_known"] = stage1_df["ram_gb"].notna()
stage1_df["condition_known"] = stage1_df["condition"].notna()

In [87]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

display(
    stage1_df[known_flags].agg(["sum", "count"]).T
)

display(
    stage1_df[known_flags].mean().mul(100).round(2)
    .to_frame("known_percentage")
)

,sum,count
storage_known,367,5258
ram_known,112,5258
condition_known,283,5258


,known_percentage
storage_known,6.98
ram_known,2.13
condition_known,5.38


In [88]:
print("Shape:", stage1_df.shape)
print("Unique ad IDs:", stage1_df["ad_id"].nunique())
print("Unique title groups:", stage1_df["title_group"].nunique())

Shape: (5258, 39)
Unique ad IDs: 5258
Unique title groups: 2191


In [89]:
display(stage1_df.dtypes.to_frame("dtype"))

,dtype
title,str
price,float64
location,str
ad_url,str
ad_id,int64
run_id,str
brand,str
storage_gb,float64
ram_gb,float64
storage_src,str


In [90]:
stage1_df['brand'].value_counts()

brand
apple       2416
samsung      984
xiaomi       380
oppo         325
realme       187
honor        181
infinix      144
vivo         131
huawei       114
motorola      24
oneplus       18
google        17
zte           17
tecno         15
sony           8
itel           7
nokia          5
lenovo         4
lg             2
Name: count, dtype: int64

In [91]:
stage1_df['model_key'].value_counts()

model_key
unknown                    562
apple iphone 13            329
apple iphone 13 pro max    175
apple iphone 11 pro max    140
apple iphone 12 pro max    127
                          ... 
google pixel 6               1
xiaomi redmi a5              1
honor magic 5 lite           1
google pixel 9 pro fold      1
honor robot                  1
Name: count, Length: 728, dtype: int64

In [92]:
# Create an independent copy for Stage 2
stage2_df = stage1_df.copy(deep=True)
print("Stage 1 shape:", stage1_df.shape)
print("Stage 2 shape:", stage2_df.shape)
print("Are they the same object?", stage1_df is stage2_df)

Stage 1 shape: (5258, 39)
Stage 2 shape: (5258, 39)
Are they the same object? False


In [93]:
categorical_features = ["model_key", "brand"]

known_features = [
    "storage_known",
    "ram_known",
    "condition_known",
]

feature_columns = categorical_features + known_features
train_df = stage2_df.loc[stage2_df["price"].notna()].copy()
X = train_df[feature_columns].copy()
y = train_df["price"].copy()
groups = train_df["title_group"].copy()

In [94]:
print("Training rows:", len(train_df))
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Missing target values:", y.isna().sum())
print("Missing title groups:", groups.isna().sum())

display(X.head())

Training rows: 5258
Features shape: (5258, 5)
Target shape: (5258,)
Missing target values: 0
Missing title groups: 0


,model_key,brand,storage_known,ram_known,condition_known
0,honor magic v5,honor,False,False,False
1,xiaomi 17 ultra,xiaomi,False,False,False
2,apple iphone 17 pro,apple,False,False,False
3,honor 500 pro,honor,False,False,False
4,apple iphone 15 pro max,apple,False,False,False


In [95]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("known_flags", "passthrough", known_features)
])

print("Preprocessor is ready.")
print("Categorical features:", categorical_features)
print("Known flags:", known_features)

Preprocessor is ready.
Categorical features: ['model_key', 'brand']
Known flags: ['storage_known', 'ram_known', 'condition_known']


In [96]:
# Check required columns
missing_columns = [col for col in feature_columns if col not in stage1_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Check that X, y, and groups have matching rows
if not (X.index.equals(y.index) and X.index.equals(groups.index)):
    raise ValueError("X, y, and groups have mismatched rows")

# Review the data
print("Features:", X.columns.tolist())
print("\nMissing values:\n", X.isna().sum())
print("\nTarget summary:\n", y.describe())
print("\nUnique title groups:", groups.nunique())

Features: ['model_key', 'brand', 'storage_known', 'ram_known', 'condition_known']

Missing values:
 model_key            0
brand              279
storage_known        0
ram_known            0
condition_known      0
dtype: int64

Target summary:
 count      5258.000000
mean      22858.786421
std       20722.258158
min        1000.000000
25%        7000.000000
50%       16500.000000
75%       32000.000000
max      185500.000000
Name: price, dtype: float64

Unique title groups: 2191


In [97]:
X_transformed = preprocessor.fit_transform(X)

print("Original shape:", X.shape)
print("Encoded shape:", X_transformed.shape)
print("Encoding successful!")

Original shape: (5258, 5)
Encoded shape: (5258, 751)
Encoding successful!


In [98]:
# Inspect model keys by brand

for brand in stage2_df["brand"].dropna().unique():

    print(f"\n===== {brand.upper()} =====")

    models = (
        stage2_df.loc[
            stage2_df["brand"] == brand,
            "model_key"
        ]
        .dropna()
        .unique()
    )

    print(models[:40])


===== HONOR =====
<StringArray>
[   'honor magic v5',     'honor 500 pro',      'honor win rt',
         'honor 600', 'honor magic 8 pro',         'honor 400',
    'honor 400 lite',         'honor x8c',         'honor x9b',
         'honor x9c',         'honor x7b',      'honor x5plus',
     'honor 200 pro',    'honor magic v2',    'honor magic v3',
          'honor x9',          'honor 8c',          'honor x8',
         'honor x6c',          'honor 50',           'unknown',
        'honor play',         'honor win',          'honor x6',
         'honor x9d', 'honor magic 6 pro',         'honor x6b',
 'honor magic 7 pro',          'honor 10',     'honor 600 pro',
     'honor magic 7',     'honor 400 pro',         'honor x7d',
     'honor magic v', 'honor magic 4 pro',         'honor 500',
         'honor x7c',    'honor magic v6',         'honor x7a',
 'honor magic 5 pro']
Length: 40, dtype: str

===== XIAOMI =====
<StringArray>
[               'xiaomi 17 ultra',                 'xiao

<StringArray>
['google pixel 11 pro fold',           'google pixel 8',
           'google pixel 7',       'google pixel 6 pro',
          'google pixel 8a',          'google pixel 9a',
          'google pixel 6a',       'google pixel 10 5g',
       'google pixel 9 pro',       'google pixel 8 pro',
           'google pixel 6',  'google pixel 9 pro fold']
Length: 12, dtype: str

===== VIVO =====
<StringArray>
[       'vivo x fold 6',              'unknown',             'vivo y31',
         'vivo x200 fe',             'vivo y91',            'vivo y12s',
             'vivo v40',             'vivo v70',             'vivo y29',
             'vivo y40',             'vivo s19',            'vivo v23e',
      'vivo x300 ultra',             'vivo y05',     'vivo v60 lite 5g',
             'vivo v21',            'vivo y11d',        'vivo y19s pro',
        'vivo x fold 3',        'vivo v60 lite',        'vivo v40 lite',
            'vivo y31d',             'vivo y50',             'vivo y15',
     

In [99]:

def extract_family(model_key, brand):

    # Handle missing or unknown values
    if pd.isna(model_key) or str(model_key).lower().strip() == "unknown":
        return "unknown"

    if pd.isna(brand):
        return "unknown"

    brand = str(brand).lower().strip()
    model = str(model_key).lower().strip()

    # Remove brand name from model_key
    model = model.removeprefix(brand).strip()

    if not model or model == "unknown":
        return "unknown"

    # Apple
    if brand == "apple":
        return "iphone"

    # Samsung
    elif brand == "samsung":
        if model.startswith("s"):
            return "galaxy s"
        elif model.startswith("a"):
            return "galaxy a"
        elif model.startswith("z trifold"):
            return "galaxy z trifold"
        elif model.startswith("z fold"):
            return "galaxy z fold"
        elif model.startswith("z flip"):
            return "galaxy z flip"
        elif model.startswith("note"):
            return "galaxy note"
        elif model.startswith("j"):
            return "galaxy j"
        elif model.startswith("grand"):
            return "galaxy grand"
        elif model.startswith("mega"):
            return "galaxy mega"
        elif model.startswith("m"):
            return "galaxy m"
        elif model.startswith("xcover"):
            return "galaxy xcover"

    # Xiaomi
    elif brand == "xiaomi":
        if model.startswith("redmi note"):
            return "redmi note"
        elif model.startswith("redmi k"):
            return "redmi k"
        elif model.startswith("redmi"):
            return "redmi"
        elif model.startswith("poco"):
            return "poco"
        else:
            return "xiaomi"

    # Oppo
    elif brand == "oppo":
        for family in ["reno", "find", "a", "f", "r"]:
            if model.startswith(family):
                return family

    # Google
    elif brand == "google":
        return "pixel"

    # Honor
    elif brand == "honor":
        for family in ["magic", "play", "robot", "win", "x", "8c"]:
            if model.startswith(family):
                return family

        if model.split()[0].isdigit():
            return "honor numeric"

    # Vivo
    elif brand == "vivo":
        for family in ["x fold", "iqoo", "x", "y", "v", "s"]:
            if model.startswith(family):
                return family

    # Realme
    elif brand == "realme":
        for family in ["gt", "narzo", "note", "a", "v", "c", "p"]:
            if model.startswith(family):
                return family

        if model[0].isdigit():
            return "realme numeric"

    # Infinix
    elif brand == "infinix":
        for family in ["smart", "note", "hot", "gt", "s"]:
            if model.startswith(family):
                return family

    # Tecno
    elif brand == "tecno":
        for family in ["spark", "camon", "pova", "pop"]:
            if model.startswith(family):
                return family

    # OnePlus
    elif brand == "oneplus":
        for family in ["nord", "ace"]:
            if model.startswith(family):
                return family

        return "oneplus"

    # Huawei
    elif brand == "huawei":
        for family in ["mate", "nova", "pura", "p", "y"]:
            if model.startswith(family):
                return family

    # Motorola
    elif brand == "motorola":
        for family in ["razr", "signature", "stylus", "edge"]:
            if model.startswith(family):
                return family

        if model.startswith("moto g") or model.startswith("g"):
            return "moto g"

    # ZTE
    elif brand == "zte":
        for family in ["red magic", "nubia", "blade"]:
            if model.startswith(family):
                return family

    # Sony
    elif brand == "sony":
        if model.startswith("xperia"):
            return "xperia"
        elif model.startswith("st"):
            return "st"

    # Itel
    elif brand == "itel":
        if model.startswith("a"):
            return "a"
        elif model.startswith("rs"):
            return "rs"

        return "other"

    # Lenovo
    elif brand == "lenovo":
        for family in ["note", "k", "a"]:
            if model.startswith(family):
                return family

    # Nokia
    elif brand == "nokia":
        for family in ["x", "n"]:
            if model.startswith(family):
                return family

    # LG
    elif brand == "lg":
        if model.startswith("g"):
            return "g"

        return "other"

    return "other"


In [100]:
stage2_df["family"] = stage2_df.apply(
    lambda row: extract_family(row["model_key"], row["brand"]),
    axis=1
)

print(stage2_df[["brand", "model_key", "family"]].head(30))

      brand                 model_key            family
0     honor            honor magic v5             magic
1    xiaomi           xiaomi 17 ultra            xiaomi
2     apple       apple iphone 17 pro            iphone
3     honor             honor 500 pro     honor numeric
4     apple   apple iphone 15 pro max            iphone
5     apple          apple iphone 16e            iphone
6      oppo              oppo find x5              find
7   samsung         samsung s24 ultra          galaxy s
8     apple   apple iphone 13 pro max            iphone
9     honor            honor magic v5             magic
10    apple           apple iphone 15            iphone
11      NaN                   unknown           unknown
12    apple   apple iphone 17 pro max            iphone
13    honor              honor win rt               win
14   google  google pixel 11 pro fold             pixel
15    apple   apple iphone 16 pro max            iphone
16   xiaomi            xiaomi 18 fold           

In [101]:
print(stage2_df["family"].value_counts())

family
iphone      2373
unknown      563
galaxy a     501
galaxy s     300
reno         165
            ... 
r              1
n              1
pop            1
g              1
robot          1
Name: count, Length: 68, dtype: int64


In [102]:
print("Other count:", (stage2_df["family"] == "other").sum())

print("\nRemaining other models:")
print(
    stage2_df.loc[
        stage2_df["family"] == "other",
        ["brand", "model_key"]
    ].drop_duplicates().to_string(index=False)
)

Other count: 0

Remaining other models:
Empty DataFrame
Columns: [brand, model_key]
Index: []


In [103]:
print(stage2_df["family"].value_counts())

family
iphone      2373
unknown      563
galaxy a     501
galaxy s     300
reno         165
            ... 
r              1
n              1
pop            1
g              1
robot          1
Name: count, Length: 68, dtype: int64


In [104]:
family_counts = stage2_df["family"].value_counts()

print(family_counts[family_counts <= 5])

family
win                 5
ace                 5
spark               5
xperia              4
camon               4
x fold              3
redmi k             3
red magic           3
nord                3
edge                2
blade               2
stylus              2
play                2
pova                2
galaxy xcover       2
k                   2
narzo               2
iqoo                2
razr                2
galaxy grand        2
signature           2
galaxy z trifold    1
st                  1
galaxy mega         1
8c                  1
rs                  1
r                   1
n                   1
pop                 1
g                   1
robot               1
Name: count, dtype: int64


In [105]:
small_families = (
    stage2_df.groupby(["brand", "family"])
    .size()
    .reset_index(name="count")
)

print(
    small_families[small_families["count"] <= 5]
    .sort_values(["brand", "count", "family"])
    .to_string(index=False)
)

   brand           family  count
   honor               8c      1
   honor            robot      1
   honor             play      2
   honor              win      5
 infinix                s      3
    itel               rs      1
  lenovo                a      1
  lenovo             note      1
  lenovo                k      2
      lg                g      1
      lg          unknown      1
motorola             edge      2
motorola             razr      2
motorola        signature      2
motorola           stylus      2
   nokia                n      1
   nokia          unknown      2
   nokia                x      2
 oneplus             nord      3
 oneplus              ace      5
    oppo                r      1
  realme                a      1
  realme            narzo      2
  realme          unknown      2
  realme                v      2
  realme                p      5
 samsung      galaxy mega      1
 samsung galaxy z trifold      1
 samsung     galaxy grand      2
 samsung  

In [106]:
unknown_models = (
    stage2_df.loc[
        stage2_df["family"] == "unknown",
        ["brand", "model_key"]
    ]
    .value_counts()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(unknown_models.head(30).to_string(index=False))

   brand   model_key  count
 samsung     unknown     66
   apple     unknown     43
    oppo     unknown     35
    vivo     unknown     29
 infinix     unknown     28
  xiaomi     unknown     27
  huawei     unknown     23
   honor     unknown     13
motorola     unknown      6
   tecno     unknown      3
    sony     unknown      3
   nokia     unknown      2
     zte     unknown      2
  realme     unknown      2
      lg     unknown      1
     zte zte unknown      1


In [107]:
print("Missing family values:", stage2_df["family"].isna().sum())
print("Total rows:", len(stage2_df))
print("Unique families:", stage2_df["family"].nunique())

Missing family values: 0
Total rows: 5258
Unique families: 68


In [108]:
print(
    stage2_df.groupby(["brand", "family"])["model_key"]
    .nunique()
    .sort_values(ascending=False)
    .to_string()
)

brand     family          
samsung   galaxy a            59
apple     iphone              54
oppo      reno                41
          a                   39
xiaomi    redmi note          34
samsung   galaxy s            33
xiaomi    xiaomi              31
          redmi               29
realme    realme numeric      25
          c                   23
infinix   hot                 21
vivo      y                   19
xiaomi    poco                18
huawei    y                   17
honor     x                   17
vivo      v                   14
honor     honor numeric       14
          magic               12
google    pixel               12
infinix   note                11
huawei    p                   11
          nova                11
samsung   galaxy z fold        9
          galaxy m             9
realme    gt                   9
samsung   galaxy note          9
huawei    mate                 9
infinix   smart                8
oppo      f                    6
oneplus   oneplu

In [109]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "samsung",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

              model_key           family
            samsung a01         galaxy a
            samsung a02         galaxy a
            samsung a03         galaxy a
       samsung a03 core         galaxy a
           samsung a03s         galaxy a
           samsung a04e         galaxy a
           samsung a04s         galaxy a
            samsung a05         galaxy a
           samsung a05s         galaxy a
            samsung a06         galaxy a
            samsung a07         galaxy a
            samsung a10         galaxy a
            samsung a11         galaxy a
            samsung a12         galaxy a
            samsung a13         galaxy a
            samsung a14         galaxy a
            samsung a15         galaxy a
            samsung a16         galaxy a
            samsung a17         galaxy a
            samsung a20         galaxy a
           samsung a20s         galaxy a
            samsung a21         galaxy a
           samsung a21s         galaxy a
            sams

In [110]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "xiaomi",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

                       model_key     family
                         unknown    unknown
                       xiaomi 11     xiaomi
                  xiaomi 11 lite     xiaomi
                 xiaomi 11 ultra     xiaomi
                       xiaomi 12     xiaomi
                   xiaomi 12 pro     xiaomi
                  xiaomi 13 lite     xiaomi
                 xiaomi 13 ultra     xiaomi
                      xiaomi 13t     xiaomi
                  xiaomi 13t pro     xiaomi
                       xiaomi 14     xiaomi
                   xiaomi 14 pro     xiaomi
                  xiaomi 14t pro     xiaomi
                       xiaomi 15     xiaomi
                   xiaomi 15 pro     xiaomi
                      xiaomi 15t     xiaomi
                       xiaomi 17     xiaomi
                 xiaomi 17 ultra     xiaomi
                      xiaomi 17t     xiaomi
                  xiaomi 17t pro     xiaomi
                  xiaomi 18 fold     xiaomi
        xiaomi black shark 4 pro

In [111]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "apple",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

              model_key  family
        apple iphone 11  iphone
    apple iphone 11 pro  iphone
apple iphone 11 pro max  iphone
        apple iphone 12  iphone
   apple iphone 12 mini  iphone
    apple iphone 12 pro  iphone
apple iphone 12 pro max  iphone
        apple iphone 13  iphone
   apple iphone 13 mini  iphone
    apple iphone 13 pro  iphone
apple iphone 13 pro max  iphone
        apple iphone 14  iphone
    apple iphone 14 max  iphone
   apple iphone 14 plus  iphone
    apple iphone 14 pro  iphone
apple iphone 14 pro max  iphone
        apple iphone 15  iphone
   apple iphone 15 plus  iphone
    apple iphone 15 pro  iphone
apple iphone 15 pro max  iphone
        apple iphone 16  iphone
   apple iphone 16 plus  iphone
    apple iphone 16 pro  iphone
apple iphone 16 pro max  iphone
       apple iphone 16e  iphone
        apple iphone 17  iphone
    apple iphone 17 air  iphone
   apple iphone 17 mini  iphone
    apple iphone 17 pro  iphone
apple iphone 17 pro max  iphone
       a

In [112]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "oppo",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

            model_key  family
             oppo a12       a
             oppo a15       a
             oppo a16       a
            oppo a16k       a
       oppo a16pro 5g       a
             oppo a18       a
             oppo a1k       a
              oppo a3       a
             oppo a31       a
             oppo a38       a
             oppo a3s       a
             oppo a3x       a
              oppo a5       a
             oppo a52       a
             oppo a53       a
             oppo a57       a
             oppo a58       a
             oppo a5i       a
           oppo a5pro       a
             oppo a5s       a
              oppo a6       a
           oppo a6 5g       a
             oppo a60       a
             oppo a6i       a
           oppo a6pro       a
        oppo a6pro 5g       a
             oppo a6x       a
              oppo a7       a
             oppo a74       a
             oppo a76       a
             oppo a77       a
             oppo a78       a
          

In [113]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "vivo",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

           model_key  family
             unknown unknown
vivo iqoo neo 10 pro    iqoo
 vivo iqoo neo 7 pro    iqoo
            vivo s19       s
        vivo s19 pro       s
            vivo s20       s
            vivo v21       v
            vivo v23       v
           vivo v23e       v
           vivo v27e       v
        vivo v29 pro       v
            vivo v40       v
       vivo v40 lite       v
            vivo v50       v
       vivo v50 lite       v
    vivo v50 lite 5g       v
            vivo v60       v
       vivo v60 lite       v
    vivo v60 lite 5g       v
            vivo v70       v
       vivo x fold 3  x fold
   vivo x fold 3 pro  x fold
       vivo x fold 6  x fold
        vivo x200 fe       x
       vivo x200 pro       x
     vivo x200 ultra       x
       vivo x300 pro       x
     vivo x300 ultra       x
            vivo y05       y
           vivo y11d       y
           vivo y12s       y
            vivo y15       y
           vivo y17s       y
            vi

In [114]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "huawei",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

               model_key  family
     huawei mate 10 lite    mate
      huawei mate 20 pro    mate
      huawei mate 40 pro    mate
      huawei mate 50 pro    mate
      huawei mate 70 pro    mate
           huawei mate 8    mate
      huawei mate 80 pro    mate
          huawei mate x6    mate
huawei mate xt2 ultimate    mate
         huawei nova 12i    nova
          huawei nova 14    nova
       huawei nova 15max    nova
          huawei nova 3i    nova
           huawei nova 5    nova
           huawei nova 7    nova
          huawei nova 7i    nova
         huawei nova 7se    nova
           huawei nova 9    nova
         huawei nova 9se    nova
         huawei nova max    nova
         huawei p10 lite       p
         huawei p20 lite       p
          huawei p20 pro       p
         huawei p30 lite       p
          huawei p30 pro       p
          huawei p40 pro       p
              huawei p50       p
          huawei p50 pro       p
               huawei p6       p
          

In [115]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "honor",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

         model_key        family
          honor 10 honor numeric
         honor 200 honor numeric
     honor 200 pro honor numeric
          honor 30 honor numeric
         honor 400 honor numeric
    honor 400 lite honor numeric
     honor 400 pro honor numeric
          honor 50 honor numeric
         honor 500 honor numeric
     honor 500 pro honor numeric
         honor 600 honor numeric
     honor 600 pro honor numeric
          honor 8c            8c
          honor 90 honor numeric
     honor 90 lite honor numeric
 honor magic 4 pro         magic
honor magic 5 lite         magic
 honor magic 5 pro         magic
 honor magic 6 pro         magic
     honor magic 7         magic
 honor magic 7 pro         magic
 honor magic 8 pro         magic
     honor magic v         magic
    honor magic v2         magic
    honor magic v3         magic
    honor magic v5         magic
    honor magic v6         magic
        honor play          play
       honor robot         robot
         h

In [116]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "motorola",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

              model_key    family
     motorola edge 2023      edge
motorola edge 60 fusion      edge
           motorola g04    moto g
      motorola moto g35    moto g
  motorola moto g5 plus    moto g
      motorola moto g54    moto g
      motorola moto g56    moto g
      motorola moto g85    moto g
 motorola razr 50 ultra      razr
 motorola razr 60 ultra      razr
     motorola signature signature
     motorola stylus 5g    stylus
                unknown   unknown


In [117]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "infinix",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

              model_key  family
      infinix gt 20 pro      gt
      infinix gt 30 pro      gt
      infinix gt 50 pro      gt
         infinix hot 10     hot
    infinix hot 10 play     hot
        infinix hot 10s     hot
         infinix hot 11     hot
    infinix hot 11 play     hot
        infinix hot 11s     hot
         infinix hot 30     hot
          infinix hot 4     hot
        infinix hot 40i     hot
         infinix hot 50     hot
        infinix hot 50i     hot
          infinix hot 6     hot
      infinix hot 6 pro     hot
         infinix hot 60     hot
     infinix hot 60 pro     hot
infinix hot 60 pro plus     hot
        infinix hot 60i     hot
         infinix hot 70     hot
          infinix hot 8     hot
          infinix hot 9     hot
     infinix hot 9 play     hot
        infinix note 10    note
     infinix note 10pro    note
        infinix note 11    note
        infinix note 40    note
        infinix note 50    note
       infinix note 50s    note
        

In [118]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "tecno",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

              model_key  family
         tecno camon 30   camon
         tecno camon 40   camon
  tecno camon 40 pro 5g   camon
            tecno pop 7     pop
       tecno pova 6 pro    pova
           tecno pova 7    pova
         tecno spark 20   spark
tecno spark 20 pro plus   spark
        tecno spark 30c   spark
         tecno spark cm   spark
       tecno spark go 3   spark
                unknown unknown


In [119]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "realme",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

         model_key         family
          realme 1 realme numeric
         realme 10 realme numeric
     realme 10 pro realme numeric
         realme 11 realme numeric
     realme 11 pro realme numeric
         realme 12 realme numeric
     realme 12 pro realme numeric
realme 12 pro plus realme numeric
         realme 13 realme numeric
     realme 13 pro realme numeric
         realme 14 realme numeric
     realme 14 pro realme numeric
realme 14 pro plus realme numeric
        realme 14t realme numeric
         realme 15 realme numeric
     realme 15 pro realme numeric
        realme 15t realme numeric
realme 16 pro plus realme numeric
          realme 5 realme numeric
          realme 6 realme numeric
         realme 6i realme numeric
          realme 7 realme numeric
          realme 8 realme numeric
      realme 8 pro realme numeric
      realme 9 pro realme numeric
     realme a7 pro              a
      realme c100i              c
        realme c11              c
        realme

In [120]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "oneplus",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

          model_key  family
     oneplus 10 pro oneplus
         oneplus 11 oneplus
        oneplus 12r oneplus
        oneplus 13t oneplus
         oneplus 15 oneplus
          oneplus 5 oneplus
      oneplus ace 5     ace
      oneplus ace 6     ace
oneplus ace 6 ultra     ace
     oneplus ace 6t     ace
     oneplus nord 3    nord
   oneplus nord ce4    nord


In [121]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "zte",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

           model_key    family
             unknown   unknown
       zte blade a36     blade
    zte blade a76 5g     blade
       zte nubia a36     nubia
    zte nubia a76 5g     nubia
     zte nubia neo 2     nubia
       zte nubia v60     nubia
   zte nubia z30 pro     nubia
zte red magic 10 air red magic
zte red magic 10 pro red magic
     zte red magic 7 red magic
         zte unknown   unknown


In [122]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "google",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

               model_key family
      google pixel 10 5g  pixel
google pixel 11 pro fold  pixel
          google pixel 6  pixel
      google pixel 6 pro  pixel
         google pixel 6a  pixel
          google pixel 7  pixel
          google pixel 8  pixel
      google pixel 8 pro  pixel
         google pixel 8a  pixel
      google pixel 9 pro  pixel
 google pixel 9 pro fold  pixel
         google pixel 9a  pixel


In [123]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "sony",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

           model_key  family
          sony st15i      st
       sony xperia 1  xperia
sony xperia 1 mark 7  xperia
   sony xperia pro-i  xperia
 sony xperia z ultra  xperia
             unknown unknown


In [124]:
print(
    stage2_df.loc[
        stage2_df["brand"] == "nokia",
        ["model_key", "family"]
    ]
    .drop_duplicates()
    .sort_values("model_key")
    .to_string(index=False)
)

 model_key  family
  nokia n9       n
  nokia x2       x
nokia x350       x
   unknown unknown


In [125]:
print("Missing families:", stage2_df["family"].isna().sum())
print("Total rows:", len(stage2_df))
print("Unique families:", stage2_df["family"].nunique())
print("\nFamily counts:")
print(stage2_df["family"].value_counts().to_string())

Missing families: 0
Total rows: 5258
Unique families: 68

Family counts:
family
iphone              2373
unknown              563
galaxy a             501
galaxy s             300
reno                 165
redmi note           141
a                    112
c                     89
redmi                 88
x                     78
y                     72
xiaomi                66
realme numeric        66
honor numeric         62
hot                   62
poco                  55
v                     43
galaxy note           34
note                  32
galaxy z fold         30
magic                 27
galaxy m              27
gt                    24
nova                  23
p                     20
pixel                 17
mate                  17
smart                 16
galaxy z flip         13
find                  10
moto g                10
f                     10
oneplus               10
s                     10
nubia                  9
pura                   7
galaxy j            

In [126]:
stage2_df.loc[
    (stage2_df["brand"] == "honor") &
    (stage2_df["model_key"] == "honor 8c"),
    "family"
] = "honor numeric"

print(
    stage2_df.loc[
        stage2_df["model_key"] == "honor 8c",
        ["model_key", "family"]
    ].drop_duplicates()
)

     model_key         family
1165  honor 8c  honor numeric


In [127]:
print("Missing families:", stage2_df["family"].isna().sum())
print("Total rows:", len(stage2_df))
print("Unique families:", stage2_df["family"].nunique())
print("\nRemaining 'other':", (stage2_df["family"] == "other").sum())
print("\nUpdated Honor families:")
print(
    stage2_df.loc[
        stage2_df["brand"] == "honor",
        "family"
    ].value_counts().to_string()
)

Missing families: 0
Total rows: 5258
Unique families: 67

Remaining 'other': 0

Updated Honor families:
family
x                70
honor numeric    63
magic            27
unknown          13
win               5
play              2
robot             1


In [128]:
stage2_df = stage2_df.copy()

print("Stage 2 shape:", stage2_df.shape)
print("Family feature ready:", "family" in stage2_df.columns)

Stage 2 shape: (5258, 40)
Family feature ready: True


In [129]:
print(categorical_features)

['model_key', 'brand']


In [130]:
categorical_features = ['model_key', 'brand', 'family']

print("Categorical features:", categorical_features)

Categorical features: ['model_key', 'brand', 'family']


In [131]:
feature_columns = categorical_features + known_features

print("Feature columns:")
print(feature_columns)

Feature columns:
['model_key', 'brand', 'family', 'storage_known', 'ram_known', 'condition_known']


In [132]:
print(preprocessor)

ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['model_key', 'brand']),
                                ('known_flags', 'passthrough',
                                 ['storage_known', 'ram_known',
                                  'condition_known'])])


In [133]:
preprocessor.transformers[0] = (
    'categorical',
    preprocessor.transformers[0][1],
    ['model_key', 'brand', 'family']
)

In [134]:
preprocessor.transformers

[('categorical',
  Pipeline(steps=[('imputer',
                   SimpleImputer(fill_value='Unknown', strategy='constant')),
                  ('encoder', OneHotEncoder(handle_unknown='ignore'))]),
  ['model_key', 'brand', 'family']),
 ('known_flags',
  'passthrough',
  ['storage_known', 'ram_known', 'condition_known'])]

In [135]:
print(X.columns.tolist())

['model_key', 'brand', 'storage_known', 'ram_known', 'condition_known']


In [137]:
print('family' in stage2_df.columns)
print(stage2_df.columns.tolist())

True
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'model_key', 'match_rule', 'confidence', 'storage_known', 'ram_known', 'condition_known', 'family']


In [138]:
train_df = stage2_df.loc[stage2_df["price"].notna()].copy()

X = train_df[feature_columns].copy()
y = train_df["price"].copy()
groups = train_df["title_group"].copy()

In [139]:
print("X shape:", X.shape)
print("Features:", X.columns.tolist())

X shape: (5258, 6)
Features: ['model_key', 'brand', 'family', 'storage_known', 'ram_known', 'condition_known']


In [142]:
# 1. Define features
categorical_features = [
    "model_key",
    "brand",
    "family"
]

known_features = [
    "storage_known",
    "ram_known",
    "condition_known"
]

feature_columns = categorical_features + known_features

# 2. Prepare data
train_df = stage2_df.loc[
    stage2_df["price"].notna()
].copy()

X = train_df[feature_columns].copy()

# 3. Define categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# 4. Build feature preprocessor
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("known_flags", "passthrough", known_features)
])

# 5. Transform features
X_transformed = preprocessor.fit_transform(X)

print("Original shape:", X.shape)
print("Encoded shape:", X_transformed.shape)
print("Features:", feature_columns)
print("Feature Pipeline completed successfully!")

Original shape: (5258, 6)
Encoded shape: (5258, 818)
Features: ['model_key', 'brand', 'family', 'storage_known', 'ram_known', 'condition_known']
Feature Pipeline completed successfully!
